In [ ]:
# 0. 환경 설정
!pip install torch torchtext datasets spacy
!python -m spacy download de_core_news_sm
!python -m spacy download en_core_web_sm

In [1]:
# 1. 기본 설정
import math
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from datasets import load_dataset
import spacy


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BATCH_SIZE = 64
EMB_SIZE = 512
NHEAD = 8
FFN_HID_DIM = 512
NUM_ENCODER_LAYERS = 3
NUM_DECODER_LAYERS = 3
DROPOUT = 0.1
MAX_LEN = 128

/Users/iwon-yong/.pyenv/versions/venv310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 2. 토크나이저 & vocab
spacy_de = spacy.load("de_core_news_sm")
spacy_en = spacy.load("en_core_web_sm")

def tokenize_de(text):
    return [tok.text.lower() for tok in spacy_de.tokenizer(text)]

def tokenize_en(text):
    return [tok.text.lower() for tok in spacy_en.tokenizer(text)]

SPECIALS = ["<pad>", "<bos>", "<eos>", "<unk>"]
PAD_IDX, BOS_IDX, EOS_IDX, UNK_IDX = range(4)


def build_vocab(dataset, tokenizer):
    vocab = {tok: idx for idx, tok in enumerate(SPECIALS)}
    idx = len(SPECIALS)

    for item in dataset:
        for tok in tokenizer(item):
            if tok not in vocab:
                vocab[tok] = idx
                idx += 1
    return vocab


dataset = load_dataset("multi30k", "de-en", split="train")

SRC_VOCAB = build_vocab(
    (x["translation"]["de"] for x in dataset), tokenize_de
)
TGT_VOCAB = build_vocab(
    (x["translation"]["en"] for x in dataset), tokenize_en
)

SRC_VOCAB_SIZE = len(SRC_VOCAB)
TGT_VOCAB_SIZE = len(TGT_VOCAB)

/Users/iwon-yong/.pyenv/versions/venv310/lib/python3.10/site-packages/spacy/util.py:910: UserWarning: [W095] Model 'de_core_news_sm' (3.8.0) was trained with spaCy v3.8.0 and may not be 100% compatible with the current version (3.7.2). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)
/Users/iwon-yong/.pyenv/versions/venv310/lib/python3.10/site-packages/spacy/util.py:910: UserWarning: [W095] Model 'en_core_web_sm' (3.8.0) was trained with spaCy v3.8.0 and may not be 100% compatible with the current version (3.7.2). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)


FileNotFoundError: Couldn't find a dataset script at /Users/iwon-yong/dev/workspace/ai/pytorchtrf/07장 트랜스포머/multi30k/multi30k.py or any data file in the same directory. Couldn't find 'multi30k' on the Hugging Face Hub either: FileNotFoundError: Dataset 'multi30k' doesn't exist on the Hub. If the repo is private or gated, make sure to log in with `huggingface-cli login`.